# SeedSec Rwanda: Maize Seed Vigor Detection Training Notebook
This Jupyter notebook sets up the workspace, downloads, unzips, structures, and trains a **YOLOv8** model for Maize Seed Vigor Stage Classification/Object Detection using the Kaggle dataset uploaded to Google Drive.

### Workflow:
1. **Google Drive Mount & API Key Configuration**
2. **Dependencies & Environment Setup** (install PyTorch, ultralytics)
3. **Dataset Verification & Structuring** (unpack and parse XML annotations to YOLO `.txt` format)
4. **Data Config Yaml Creation**
5. **Model Training** (YOLOv8 Training)
6. **Model Verification & Testing**
7. **TFLite Export** (for offline edge execution)

## Step 1: Set Kaggle Token & Mount Google Drive

In [ ]:
import os
from google.colab import drive

# Set Kaggle API token
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_6ee77fd6fece02b5b52d748e8'

# Mount Google Drive
drive.mount('/content/drive')

## Step 2: Install Libraries & Verify Dataset Location

In [ ]:
# Install Ultralytics for YOLOv8
!pip install ultralytics -q

# Verify dataset directory contents
dataset_dir = "/content/drive/MyDrive/KaggleDatasets/seed-vigor-detection"
print("Checking contents of dataset directory:")
!ls -la "$dataset_dir"

## Step 3: Convert Annotations & Organize Dataset for YOLO
YOLOv8 requires dataset splitting into `images/` and `labels/` subfolders for `train` and `val` sets, and annotations must be in Normalized YOLO format (`.txt` files containing: `<class_id> <x_center> <y_center> <width> <height>`).

This cell parses the XML files (from PASCAL VOC format) and structures the directories correctly.

In [ ]:
import xml.etree.ElementTree as ET
import glob
import shutil
import random
from pathlib import Path

# Paths configuration
source_dir = Path("/content/drive/MyDrive/KaggleDatasets/seed-vigor-detection")
yolo_dataset_dir = Path("/content/seed_vigor_yolo_format")

# Define target subdirectories
for split in ["train", "val"]:
    (yolo_dataset_dir / "images" / split).mkdir(parents=True, exist_ok=True)
    (yolo_dataset_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

# Classes mapping
class_mapping = {
    "ungerminated": 0,
    "germinated": 1,
    "vigor_1": 2,
    "vigor_2": 3,
    "vigor_3": 4
}
discovered_classes = set()

def convert_xml_to_yolo(xml_file, output_txt_path, img_width, img_height):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    size = root.find("size")
    if size is not None:
        img_width = int(size.find("width").text)
        img_height = int(size.find("height").text)
        
    lines = []
    for obj in root.findall("object"):
        class_name = obj.find("name").text.strip().lower()
        discovered_classes.add(class_name)
        
        if class_name not in class_mapping:
            class_mapping[class_name] = len(class_mapping)
        class_id = class_mapping[class_name]
        
        bbox = obj.find("bndbox")
        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)
        
        x_center = ((xmin + xmax) / 2.0) / img_width
        y_center = ((ymin + ymax) / 2.0) / img_height
        width = (xmax - xmin) / img_width
        height = (ymax - ymin) / img_height
        
        lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
        
    with open(output_txt_path, "w") as f:
        f.write("\n".join(lines))

# Gather all images and annotations
image_files = sorted(glob.glob(str(source_dir / "*.jpg")) + glob.glob(str(source_dir / "*.png")) + glob.glob(str(source_dir / "*.jpeg")))
xml_files = sorted(glob.glob(str(source_dir / "*.xml")))

print(f"Found {len(image_files)} images and {len(xml_files)} XML annotations.")

# Match image names with xml
dataset_pairs = []
for xml_path in xml_files:
    xml_stem = Path(xml_path).stem
    matching_img = None
    for ext in [".jpg", ".png", ".jpeg"]:
        img_candidate = source_dir / f"{xml_stem}{ext}"
        if img_candidate.exists():
            matching_img = img_candidate
            break
    if matching_img:
        dataset_pairs.append((matching_img, xml_path))

print(f"Paired {len(dataset_pairs)} image-annotation sets.")

# Train/Val split (80% train, 20% validation)
random.seed(42)
random.shuffle(dataset_pairs)
split_idx = int(len(dataset_pairs) * 0.8)
train_pairs = dataset_pairs[:split_idx]
val_pairs = dataset_pairs[split_idx:]

def process_pairs(pairs, split_name):
    count = 0
    for img_path, xml_path in pairs:
        dest_img_path = yolo_dataset_dir / "images" / split_name / img_path.name
        dest_lbl_path = yolo_dataset_dir / "labels" / split_name / f"{img_path.stem}.txt"
        shutil.copy(str(img_path), str(dest_img_path))
        convert_xml_to_yolo(xml_path, dest_lbl_path, 640, 640)
        count += 1
    print(f"Processed {count} entries into {split_name} subset.")

process_pairs(train_pairs, "train")
process_pairs(val_pairs, "val")
print("Discovered classes in annotations:", list(discovered_classes))
print("Final Class Mapping:", class_mapping)

## Step 4: Create YOLOv8 Config File (`dataset.yaml`)
Ultralytics YOLO uses a YAML file specifying the dataset root directory, subdirectories for training/validation subsets, classes dictionary, and number of classes.

In [ ]:
import yaml

# Construct dynamic dataset YAML structure
classes_list = sorted(class_mapping.items(), key=lambda x: x[1])
classes_names = {idx: name for name, idx in classes_list}

yaml_content = {
    "path": "/content/seed_vigor_yolo_format",
    "train": "images/train",
    "val": "images/val",
    "names": classes_names
}

yaml_file_path = "/content/seed_vigor_yolo_format/dataset.yaml"
with open(yaml_file_path, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("Created dataset.yaml content:")
with open(yaml_file_path, "r") as f:
    print(f.read())

## Step 5: Initialize and Train YOLOv8 Model
We will use YOLOv8n (YOLOv8 Nano, lightweight for later translation to mobile) loaded with pre-trained weights for transfer learning, training for 50 epochs.

In [ ]:
from ultralytics import YOLO

# Load pre-trained nano detector
model = YOLO("yolov8n.pt")

# Begin training run
results = model.train(
    data="/content/seed_vigor_yolo_format/dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/drive/MyDrive/KaggleDatasets/seed-vigor-detection/runs",
    name="seed_vigor_detection_yolov8"
)

print("Training completed! Run weights saved in your Google Drive.")

## Step 6: Validate & Export Model to TFLite (for Offline App)
Once trained, we validate the model on the validation set, verify precision and recall metrics, and export the best checkpoint to TensorFlow Lite format (with FP16 quantization).

In [ ]:
# Load the best trained model
best_weights_path = "/content/drive/MyDrive/KaggleDatasets/seed-vigor-detection/runs/seed_vigor_detection_yolov8/weights/best.pt"
best_model = YOLO(best_weights_path)

# Validate performance
metrics = best_model.val()
print("mAP50-95 score:", metrics.box.map)
print("mAP50 score:", metrics.box.map50)

# Export to TFLite (quantized FP16)
print("Exporting model to quantized TFLite...")
tflite_path = best_model.export(format="tflite", int8=False, half=True)
print(f"Model exported successfully! Download your TFLite model from: {tflite_path}")

# Copy exported weights directly to dataset workspace on Drive
import shutil
shutil.copy(tflite_path, "/content/drive/MyDrive/KaggleDatasets/seed-vigor-detection/best_seed_vigor_fp16.tflite")
print("Saved best_seed_vigor_fp16.tflite directly in drive directory.")